In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import tarfile
import glob
import pandas as pd
import re
import os
import matplotlib.pyplot as plt
from functools import lru_cache
import time

In [5]:
avail_pics = [i.split("\\")[-1] for i in glob.glob("drive/MyDrive/word_detect/data/**/*.png", recursive = True)]

In [6]:
if not os.path.isdir("drive/MyDrive/word_detect/xmlfiles"):
  file = tarfile.open("drive/MyDrive/word_detect/xml.tgz", "r")
  itr = iter(file)
  for i in range(len(file.getnames())):
        nxt = next(itr)
        #print(glob.glob("*/{}png".format(nxt.name[:-3])))
        if nxt.name[:-3]+"png" in avail_pics:
              file.extract(nxt, "./xmlfiles")

In [7]:
xmls = glob.glob("drive/MyDrive/word_detect/xmlfiles/*.xml")
from xml.etree import ElementTree as ET

In [8]:
#glob.glob("drive/MyDrive/word_detect/data/**/{}png".format(nxt.name[:-3]), recursive = True)

In [9]:
wordcount = {}
pattern = re.compile("\w+")
for name in xmls:
    parsed = ET.parse(name)
    root = parsed.getroot()
    txt = root.find("machine-printed-part")
    for item in txt:
        for word in pattern.findall(item.attrib['text']):
            if len(word) > 1:
                wordcount.setdefault(word, {})
                wordcount[word].setdefault("count", 0)
                wordcount[word].setdefault("forms", [])
                wordcount[word]["count"] += 1
                if not name in wordcount[word]["forms"]:
                  wordcount[word]["forms"].append(name)

In [10]:
wordcountDf = pd.DataFrame({"word": wordcount.keys(), "count" : [i["count"] for i in wordcount.values()], "nforms" : [i["forms"]
for i in wordcount.values()]}, index = range(len(wordcount)))

In [33]:
df = wordcountDf[(wordcountDf["count"]  >= 50) & (wordcountDf["count"]  <= 700)].sample(frac = 1)

In [25]:
@lru_cache(maxsize = 1540)
def imread(tname):
    return plt.imread(glob.glob("drive/MyDrive/word_detect/data/**/{}png".format(tname.split("/")[-1][:-3]), recursive = True)[0])

In [1]:
len(imread)

NameError: name 'imread' is not defined

In [26]:
def generateWordArr(entry):
  arrays = []
  for tname in entry["nforms"]:
    img = imread(tname)
    parsed = ET.parse(tname)
    root = parsed.getroot()
    hand = root.find("handwritten-part")
    for line in hand:
      #print(line.attrib["text"])
      for word in line.findall("word"):
          if word.attrib['text'] == entry["word"]:
            #print("found")
            coords = {}
            for letter in word:
                for key, value in letter.attrib.items():
                    coords.setdefault(key, [])
                    coords[key].append(int(value))
            xl, yl, xr, yr = min(coords["x"]), min(coords["y"]), max([coords["width"][i] + coords["x"][i]
                                                                           for i in range(len(coords["x"]))]), max([coords["height"]
                                                                                                                    [i] + coords["y"][i] for i in range(len(coords["x"]))])
            array = img[yl:yr, xl:xr]
            arrays.append(array)
  return arrays

In [42]:
def generateDf2(lis):
    arrays, labels, shapes = [], [], []
    total = df["count"].sum()
    pos, section =  0, 0
    start = time.time()
    for picname in avail_pics:
        img = plt.imread(glob.glob("data/**/{}".format(picname))[0])
        tname = glob.glob("xmlfiles/{}xml".format(picname[:-3]))[0]
        parsed = ET.parse(tname)
        root = parsed.getroot()
        hand = root.find("handwritten-part")
        for line in hand:
            for word in line.findall("word"):
                    if word.attrib['text'] in lis:
                        try:
                            coords = {}
                            for letter in word:
                                for key, value in letter.attrib.items():
                                    coords.setdefault(key, [])
                                    coords[key].append(int(value))
                            xl, yl, xr, yr = min(coords["x"]), min(coords["y"]), max([coords["width"][i] + coords["x"][i]
                                                                                           for i in range(len(coords["x"]))]), max([coords["height"]
                                                                                                                                    [i] + coords["y"][i] for i in range(len(coords["x"]))])
                            array = img[yl:yr, xl:xr]
                            arrays.append(array.flatten())
                            shapes.append(array.shape)
                            labels.append(word.attrib['text'])
                            pos += 1
                            if pos%100 == 0:
                                print("{} of {}-----{}".format(pos, total, time.time()-start))
                                start = time.time()
                            if pos%1000 == 0:
                                pd.DataFrame({"array": arrays, "shape": shapes, "label": labels}).to_csv("files/file{}.csv".format(section))
                                section += 1
                                arrays, labels, shapes = [], [], []
                        except:
                            pass
    return arrays

In [ ]:
start = time.time()
df1 = generateDF(df)
stop = time.time()

In [36]:
len(df)

191

In [ ]:
a["label"].value_counts()

tell       25
food       24
troops     21
Kennedy    20
Name: label, dtype: int64

In [ ]:
d = wordcountDf[wordcountDf["count"]  == 25][:4]
[len(i) for i in d["nforms"]]

[23, 25, 25, 24]

In [ ]:
wordcountDf[wordcountDf["count"]  == 25][:4]

,word,count,nforms
341,Kennedy,25,"[xmlfiles\a01-049.xml, xmlfiles\a01-049u.xml, ..."
455,troops,25,"[xmlfiles\a01-068u.xml, xmlfiles\a04-043.xml, ..."
740,food,25,"[xmlfiles\a01-122.xml, xmlfiles\a01-122u.xml, ..."
1008,tell,25,"[xmlfiles\a02-032.xml, xmlfiles\a05-113.xml, x..."


In [ ]:
a.to_csv("df.csv")

In [ ]:
test = pd.read_csv("df.csv")

In [ ]:
pd.test.iloc[2, 1]

'[[0.92156863 0.91764706 0.9137255  ... 0.8862745  0.90588236 0.9137255 ]\n [0.90588236 0.90588236 0.90588236 ... 0.89411765 0.9019608  0.90588236]\n [0.91764706 0.9137255  0.90588236 ... 0.89411765 0.8901961  0.8745098 ]\n ...\n [0.9490196  0.9411765  0.9490196  ... 0.8901961  0.9098039  0.9098039 ]\n [0.9372549  0.9372549  0.9490196  ... 0.9098039  0.9098039  0.91764706]\n [0.9607843  0.96862745 0.9490196  ... 0.8980392  0.8980392  0.90588236]]'

In [ ]:
arr = plt.imread(tname[:-3]+"png")

FileNotFoundError: [Errno 2] No such file or directory: 'drive/MyDrive/word_detect/e01-059.png'